# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the "Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution" dataset using the `mlcroissant` library.

### Dataset Source
The dataset's metadata and structure are described in a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access Croissant metadata as an object
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List available record sets, fields, and their IDs by referencing Croissant `@id` values.

In [ ]:
# List record sets and fields with their @id values
def list_record_sets(ds):
    print("Available Record Sets:")
    for recset in ds.record_sets:
        print(f"- Name: {getattr(recset, 'name', '<Unnamed RecordSet>')}, @id: {recset.id}")
        print("  Fields:")
        for field in getattr(recset, 'fields', []):
            print(f"    - {getattr(field, 'name', '<Unnamed Field>')} (@id: {field.id})")

list_record_sets(dataset)

## 3. Data Extraction
Load records from specific record sets into DataFrames for further analysis. Use the correct record set and field `@id`s as seen above.

In [ ]:
# Gather the list of record set @ids (update if more are present)
record_set_ids = [recset.id for recset in dataset.record_sets]

dataframes = {}
for recset_id in record_set_ids:
    # Fetch all records for the given record set
    data = list(dataset.records(record_set=recset_id))
    if data:
        dataframes[recset_id] = pd.DataFrame(data)
        print(f"Loaded records for {recset_id}. Columns: {list(dataframes[recset_id].columns)}\n")
    else:
        print(f"No records found for {recset_id}.")

# For demonstration, pick the first record set with data
main_recset_id = None
for recset_id, df in dataframes.items():
    if not df.empty:
        main_recset_id = recset_id
        break

if main_recset_id:
    print(f"Using Record Set: {main_recset_id}")
    display(dataframes[main_recset_id].head())
else:
    print("No usable record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the main dataset. We'll use only the `@id` of fields and columns in all code.

In [ ]:
import numpy as np

# Get available columns (field @ids) for reference
cols = dataframes[main_recset_id].columns.tolist()
print(f"Available columns in main record set ({main_recset_id}):\n", cols)

# Let's try to find numeric fields (by typical Croissant or clinical naming).
# For demonstration, suppose one candidate @id is 'age' or similar, else pick first numeric column.
numeric_field_id = None
for col in cols:
    if ('age' in col.lower()) or (('interval' in col.lower()) and ('days' not in col.lower())):
        numeric_field_id = col
        break

# Fallback: Try columns that look numeric by simple dtype test
if not numeric_field_id:
    for col in cols:
        if np.issubdtype(dataframes[main_recset_id][col].dtype, np.number):
            numeric_field_id = col
            break

if numeric_field_id is None:
    raise Exception("No numeric field found for analysis. Please check column names.")

print(f"Using numeric field for EDA: {numeric_field_id}")

df = dataframes[main_recset_id].copy()

# Remove outliers: keep records below 99th percentile only
if np.issubdtype(df[numeric_field_id].dtype, np.number):
    threshold = df[numeric_field_id].quantile(0.99)
    filtered_df = df[df[numeric_field_id] < threshold].copy()
    print(f"Filtered records to exclude outliers in {numeric_field_id} above {threshold:.2f}.")
else:
    filtered_df = df.copy()
    print(f"{numeric_field_id} is not numeric. Skipping outlier removal.")

# Normalize
if np.issubdtype(filtered_df[numeric_field_id].dtype, np.number):
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_col]].head())
else:
    print(f"Cannot normalize non-numeric column: {numeric_field_id}")

# Group by a categorical field: Pick first non-numeric field by @id
group_field_id = None
for col in cols:
    if not np.issubdtype(df[col].dtype, np.number):
        group_field_id = col
        break

if group_field_id:
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouped, its mean value by categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id is set, bar plot grouped means
if 'grouped_df' in locals():
    plt.figure(figsize=(8,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 clinical colorectal cancer dataset defined by a Croissant schema, using only `@id` references for all structural elements. 

We:
- Loaded dataset metadata and reviewed available record sets and fields by their `@id`s.
- Loaded records into pandas DataFrames for programmatic analysis.
- Selected a numeric field (by `@id`) for basic exploratory data analysis: outlier removal, normalization, and grouping.
- Visualized field distributions and explored category-level summaries, following best practices for dataset exploration with the `mlcroissant` library.

For details on available fields and their exact schema, refer to the official [Croissant metadata schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).
